# Toy TFT for outage recovery

This notebook is a small end-to-end test. It:

1. loads a merged county-hour dataset,
2. builds a simple time-series panel,
3. fits a baseline,
4. fits a tiny Temporal Fusion Transformer,
5. compares the two.

Expected input file: `merged_outage_weather.csv` with at least these columns:
`CountyFIPS`, `datetime`, `outageFraction`, `gust_mps`, `wind_speed_mps`, `precip_mm`, `pressure_hpa`.

Keep this small. The point is to get a first working run, not a final model.

In [ ]:
# If this import fails, install or update your environment.
try:
    import lightning.pytorch as pl
    print('Using lightning.pytorch')
except ImportError:
    import pytorch_lightning as pl
    print('Using pytorch_lightning fallback')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE

print('Imports loaded')

In [ ]:
# Paths and model settings.
FILE_PATH = 'merged_outage_weather.csv'

MAX_ENCODER_LENGTH = 72    # past 72 hours
MAX_PREDICTION_LENGTH = 12 # predict next 12 hours
BATCH_SIZE = 64
SEED = 42

pl.seed_everything(SEED)
print('Seed set to', SEED)

In [ ]:
# Load the merged table.
df = pd.read_csv(FILE_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])
df['CountyFIPS'] = df['CountyFIPS'].astype(str).str.zfill(5)
df = df.sort_values(['CountyFIPS', 'datetime']).reset_index(drop=True)

print(df.head())
print('Rows:', len(df))
print('Counties:', df['CountyFIPS'].nunique())
print('Date range:', df['datetime'].min(), 'to', df['datetime'].max())

In [ ]:
# Toy subset: keep the counties with the most outage mass.
toy_counties = (
    df.groupby('CountyFIPS')['outageFraction']
      .sum()
      .sort_values(ascending=False)
      .head(8)
      .index
)
df = df[df['CountyFIPS'].isin(toy_counties)].copy()

df['county'] = df['CountyFIPS']
df['time_idx'] = ((df['datetime'] - df.groupby('county')['datetime'].transform('min')).dt.total_seconds() // 3600).astype(int)
df['hour'] = df['datetime'].dt.hour.astype(int)
df['dayofweek'] = df['datetime'].dt.dayofweek.astype(int)
df['month'] = df['datetime'].dt.month.astype(int)

needed = [
    'county', 'time_idx', 'outageFraction',
    'gust_mps', 'wind_speed_mps', 'precip_mm', 'pressure_hpa',
    'hour', 'dayofweek', 'month'
]
df = df.dropna(subset=needed).copy()

print('Rows after cleanup:', len(df))
print('Counties in toy set:', df['county'].nunique())
print(df[needed].head())

In [ ]:
# Split for training and validation.
training_cutoff = df['time_idx'].max() - MAX_PREDICTION_LENGTH

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='outageFraction',
    group_ids=['county'],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=['county'],
    time_varying_known_reals=['time_idx', 'hour', 'dayofweek', 'month'],
    time_varying_unknown_reals=[
        'outageFraction',
        'gust_mps',
        'wind_speed_mps',
        'precip_mm',
        'pressure_hpa',
    ],
    target_normalizer=GroupNormalizer(groups=['county']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df,
    predict=True,
    stop_randomization=True,
)

train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

print('Training windows:', len(training))
print('Validation windows:', len(validation))

In [ ]:
# Baseline.
baseline = Baseline()
baseline_pred = baseline.predict(val_dataloader, return_y=True)
baseline_mae = MAE()(baseline_pred.output, baseline_pred.y)
print('Baseline MAE:', float(baseline_mae))

In [ ]:
# Tiny TFT. Keep it small for the first run.
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=8,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print('Model size:', tft.size())

In [ ]:
# Train.
trainer = pl.Trainer(
    max_epochs=5,
    accelerator='auto',
    devices='auto',
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=True,
)

trainer.fit(tft, train_dataloader, val_dataloader)

In [ ]:
# Evaluate.
pred = tft.predict(val_dataloader, return_y=True)
tft_mae = MAE()(pred.output, pred.y)
print('TFT MAE:', float(tft_mae))

In [ ]:
# Quick plot for a first check.
raw_predictions, x = tft.predict(val_dataloader, mode='raw', return_x=True)
for idx in range(min(1, len(raw_predictions))):
    fig = tft.plot_prediction(x, raw_predictions, idx=idx, add_loss_to_title=True)
    plt.show(fig)